In [ ]:
from main import sheet_processor
import logging
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json
import os
from utils.processing import DataSampler, create_report, make_json_safe
import traceback

# How to set log level to debug
logging.basicConfig(level=logging.DEBUG)

logger = logging.getLogger(__name__)

## Model Selection

In this section, we specify which models the system will use. To simplify the initial setup and testing workflow, we currently use a single selected model across all processing components: **PII Detection, PII Reflection, Non-PII Detection, and ReadMe Detection**. This approach allows us to validate the end-to-end pipeline without introducing variability from multiple model behaviors.

At the moment, only two models are deployed through the Azure service: **GPT-4.1 Mini** and **GPT-4.1 Nano**. These models are the available choices for experimentation and integration. As the project evolves, additional models can be added or different models can be assigned to specific components for more specialized behavior.


In [ ]:
MODEL = 'gpt-4.1-nano'

ALLOWED_MODELS = ['gpt-4.1-mini', 'gpt-4.1-nano']

if MODEL not in ALLOWED_MODELS:
    raise ValueError(f'Invalid model: {MODEL}. Please use one of the following: {", ".join(ALLOWED_MODELS)}')

# Dataset Selection

You can select a dataset in one of two ways:

1. **Using a download URL**: Provide the URL to a CSV, XLS, or XLSX file.
2. **Using a local file**: Choose a file from the `research/data` folder in the project.

This flexibility allows you to work with both online datasets and local files for testing and analysis.


In [ ]:
file_path = 'research/data/'  # This is a local test file

# DataSampler

The `DataSampler` class is used to extract a subset of rows from the original dataset.  
This sampling helps to:

- **Reduce memory usage** when working with large datasets.
- **Increase processing speed** during classification and analysis.

By working on a smaller, representative portion of the data, we can efficiently test the pipeline and classifiers without loading the entire dataset.


# Ground truth

This part of the code makes the sdd reports, but empty so we can later on fill in the real values


In [ ]:
# Start by making empty reports for each file as groundtruth to fill in later

sampler = DataSampler()

with open('/Users/liangtelkamp/Documents/GitHub/hdx-ssd-pipeline/data/isps.json', 'r') as f:
    isp = json.load(f)

isp = isp['default']

for file in os.listdir('research/data'):
    file_path = f'research/data/{file}'
    # If file already exists, skip
    if os.path.exists(f'research/results/test_results/groundtruth/{file}.json'):
        logger.info(f'File {file} already exists. Skipping.')
        continue
    try:
        sdd_report = create_report(file_path)
        sdd_report = make_json_safe(sdd_report)
        with open(f'research/results/test_results/groundtruth/{file}.json', 'w') as f:
            json.dump(sdd_report, f, indent=4)
        logger.info(f'Report saved to research/results/test_results/groundtruth/{file}.json')
    except Exception as e:
        logger.warning(f'Error processing file: {file} {e}')
        logger.info(f'Error details: {traceback.format_exc()}')
        break
        continue

# Processing Each Sheet Individually

For each sheet in the dataset, we perform the following processing steps:

1. **PII Detection** – Identify columns containing personally identifiable information.
2. **PII Reflection Detection** – Detect columns that might indirectly reveal PII.
3. **Non-PII Detection** – Classify remaining columns that do not contain sensitive information.

**Special Case:**  
If a sheet is named `readme`, `instructions`, or `metadata`, we skip the column-level classification and instead perform a **simple ReadMe scan** to extract relevant information from the documentation.


In [ ]:
# Start making sdd reports for a model
MODEL = 'gpt-4.1-nano'
# Start by making empty reports for each file as groundtruth to fill in later

sampler = DataSampler()

with open('/Users/liangtelkamp/Documents/GitHub/hdx-ssd-pipeline/data/isps.json', 'r') as f:
    isp = json.load(f)

isp = isp['default']

for file in os.listdir('research/data'):
    file_path = f'research/data/{file}'
    # Check if file already exists
    output_path = f'research/results/test_results/{MODEL}/{file}.json'
    if os.path.exists(output_path):
        logger.info(f'File {file} already exists, loading existing report')
        # Load the existing report
        with open(output_path, 'r') as f:
            sdd_report = json.load(f)
        logger.info(f'Report loaded from {output_path}')
    else:
        # Check if file is already processed
        if not os.path.exists(f'research/results/test_results/groundtruth/{file}.json'):
            logger.warning(f'File {file} not found in groundtruth, skipping')
            continue
        else:
            with open(f'research/results/test_results/groundtruth/{file}.json', 'r') as f:
                logger.info(f'File {file} found in groundtruth, loading')
                sdd_report = json.load(f)
    reports = []
    for sheet in sdd_report:
        sdd_report = sheet_processor(sheet, isp, MODEL)
        reports.append(sdd_report)
    with open(output_path, 'w') as f:
        json.dump(reports, f, indent=4)
    logger.info(f'Report saved to {output_path}')

# Save


# Evaluation


In [ ]:
def compare_pii_columns(gt_reports, pred_reports):
    """Compare PII sensitivity for all columns across sheets."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        gt_cols = {c['column_name']: c['pii']['sensitive'] for c in gt['columns']}
        pred_cols = {c['column_name']: c['pii']['sensitive'] for c in pred['columns']}
        for col_name in gt_cols:
            records.append({'column_name': col_name, 'true': gt_cols[col_name], 'pred': pred_cols.get(col_name, False)})
    return pd.DataFrame(records)


def compare_pii_table_level(gt_reports, pred_reports):
    """Compare PII sensitivity at the table level."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        records.append({'true': gt.get('pii_sensitive', False), 'pred': pred.get('pii_sensitive', False)})
    return pd.DataFrame(records)


def compare_non_pii_table_level(gt_reports, pred_reports):
    """Compare non-PII sensitivity at the table level."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        records.append({'true': gt.get('non_pii_sensitive', False), 'pred': pred.get('non_pii_sensitive', False)})
    return pd.DataFrame(records)


def calculate_metrics(df: pd.DataFrame):
    """Compute accuracy, precision, recall, and F1 score."""
    return {
        'accuracy': accuracy_score(df['true'], df['pred']),
        'precision': precision_score(df['true'], df['pred'], zero_division=0),
        'recall': recall_score(df['true'], df['pred'], zero_division=0),
        'f1': f1_score(df['true'], df['pred'], zero_division=0),
    }

In [ ]:
filename = 'data.xlsx'

# Read the groundtruth and predictions
with open(f'research/results/test_results/groundtruth/{filename}.json', 'r') as f:
    groundtruth = json.load(f)
with open(f'research/results/test_results/gpt-4.1-nano/{filename}.json', 'r') as f:
    predictions = json.load(f)

# Calculate metrics
metrics = {
    filename: {
        'pii_columns': calculate_metrics(compare_pii_columns(groundtruth, predictions)),
        'pii_table_level': calculate_metrics(compare_pii_table_level(groundtruth, predictions)),
        'non_pii_table_level': calculate_metrics(compare_non_pii_table_level(groundtruth, predictions)),
    }
}
metrics